In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re
from dataset import ASAPDataset

/home/hbli/songformer/env/miniforge3/envs/midi2scoretf/lib/python3.11/site-packages/pretty_midi/instrument.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
PDMX_ROOT = Path("/mnt/ssd/hbli/datasets/PDMX")
PDMX_CSV  = PDMX_ROOT / "PDMX.csv"

OUT_DIR = PDMX_ROOT / "derived_unpaired_58k"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_N = 58646
SEED = 42

print("PDMX_CSV exists:", PDMX_CSV.exists(), PDMX_CSV)
print("OUT_DIR:", OUT_DIR)

PDMX_CSV exists: True /mnt/ssd/hbli/datasets/PDMX/PDMX.csv
OUT_DIR: /mnt/ssd/hbli/datasets/PDMX/derived_unpaired_58k


In [3]:
df = pd.read_csv(PDMX_CSV)
# only df_keep valid records
df_keep = df[df['subset:no_license_conflict'] & df['subset:valid_mxl_pdf']].copy()

In [4]:
df_keep["mxl_abspath"] = PDMX_ROOT / df_keep['mxl']

### normalize PDMX title and composer

In [5]:
def norm(s: str) -> str:
    if s is None:
        return ""
    s = str(s).lower()
    s = re.sub(r"\(.*?\)", " ", s)
    s = re.sub(r"\[.*?\]", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def make_key(title, composer) -> str:
    return f"{norm(composer)}||{norm(title)}"

In [6]:
df_keep["title_norm"] = df_keep['title'].apply(norm)
df_keep["comp_norm"]  = df_keep['composer_name'].apply(norm)
df_keep["key"] = df_keep.apply(lambda r: f"{r['comp_norm']}||{r['title_norm']}", axis=1)

df_keep[["title", "title_norm", "comp_norm", "key"]].head(5)

,title,title_norm,comp_norm,key
2,Helvic Head,helvic head,nan,nan||helvic head
3,The Lady On The Island,the lady on the island,nan,nan||the lady on the island
4,JOHN ROY STEWART. a Strathspey.,john roy stewart a strathspey,nan,nan||john roy stewart a strathspey
5,Oddfellows Holiday. HS.09,oddfellows holiday hs 09,set of quadrillesno3,set of quadrillesno3||oddfellows holiday hs 09
6,Roslyn Castle,roslyn castle,nan,nan||roslyn castle


### normalize ASAP test set

In [7]:
data_dir = "/mnt/ssd/hbli/datasets/PM2S_dataset/midi2scoretransformer/"
q = ASAPDataset(data_dir, split = 'test')
df_asap_meta = q.metadata
# cleansing ASAP_ in title column
df_asap_meta['title_drop_ASAP'] = df_asap_meta['title'].apply(lambda x: x.replace("ASAP_", ""))

In [9]:
df_asap_meta["title_norm"] = df_asap_meta["title_drop_ASAP"].apply(norm)
df_asap_meta["comp_norm"]  = df_asap_meta["composer"].apply(norm)
df_asap_meta["key"] = df_asap_meta["title_norm"].radd("||").radd(df_asap_meta["comp_norm"])

df_asap_meta[["title_drop_ASAP", "title_norm", "comp_norm", "key"]].head(5)

,title_drop_ASAP,title_norm,comp_norm,key
0,Fugue_bwv_846,fugue bwv 846,bach,bach||fugue bwv 846
155,Piano_Sonatas_10-1,piano sonatas 10 1,beethoven,beethoven||piano sonatas 10 1
269,Ballades_1,ballades 1,chopin,chopin||ballades 1
270,Ballades_1,ballades 1,chopin,chopin||ballades 1
271,Ballades_1,ballades 1,chopin,chopin||ballades 1


### drop overlap

#### pattern

In [ ]:
df_asap_meta

,index,performance_id,composer,piece_id,title,source,performance_audio_external,performance_MIDI_external,MIDI_score_external,performance_annotation_external,...,aligned,performance_annotation,score_annotation,duration,split,title_drop_ASAP,title_norm,comp_norm,key,block_key
0,59,R_60,Bach,15,ASAP_Fugue_bwv_846,ASAP,{ASAP}/Bach/Fugue/bwv_846/Shi05M.wav,{ASAP}/Bach/Fugue/bwv_846/Shi05M.mid,{ASAP}/Bach/Fugue/bwv_846/midi_score.mid,{ASAP}/Bach/Fugue/bwv_846/Shi05M_annotations.txt,...,True,R_60_ASAP_annotation.tsv,ASAP_bwv_846_annotation.tsv,147.242188,train,Fugue_bwv_846,fugue bwv 846,bach,bach||fugue bwv 846,bach||fugue||846
155,214,R_215,Beethoven,78,ASAP_Piano_Sonatas_10-1,ASAP,{ASAP}/Beethoven/Piano_Sonatas/10-1/Hou02M.wav,{ASAP}/Beethoven/Piano_Sonatas/10-1/Hou02M.mid,{ASAP}/Beethoven/Piano_Sonatas/10-1/midi_score...,{ASAP}/Beethoven/Piano_Sonatas/10-1/Hou02M_ann...,...,True,R_215_ASAP_annotation.tsv,ASAP_10-1_annotation.tsv,278.940104,test,Piano_Sonatas_10-1,piano sonatas 10 1,beethoven,beethoven||piano sonatas 10 1,beethoven||sonata||10||1
269,334,R_335,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/BuiJL04M.wav,{ASAP}/Chopin/Ballades/1/BuiJL04M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/BuiJL04M_annotations.txt,...,True,R_335_ASAP_annotation.tsv,ASAP_1_annotation.tsv,558.778125,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
270,335,R_336,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/JIA06M.wav,{ASAP}/Chopin/Ballades/1/JIA06M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/JIA06M_annotations.txt,...,True,R_336_ASAP_annotation.tsv,ASAP_1_annotation.tsv,551.790625,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
271,336,R_337,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/KimSuyeon04M.wav,{ASAP}/Chopin/Ballades/1/KimSuyeon04M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/KimSuyeon04M_annotati...,...,True,R_337_ASAP_annotation.tsv,ASAP_1_annotation.tsv,528.455729,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
272,337,R_338,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/Mo07M.wav,{ASAP}/Chopin/Ballades/1/Mo07M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/Mo07M_annotations.txt,...,True,R_338_ASAP_annotation.tsv,ASAP_1_annotation.tsv,590.019531,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
273,339,R_340,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/Richardson13M.wav,{ASAP}/Chopin/Ballades/1/Richardson13M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/Richardson13M_annotat...,...,True,R_340_ASAP_annotation.tsv,ASAP_1_annotation.tsv,511.024740,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
274,340,R_341,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/Sladek04M.wav,{ASAP}/Chopin/Ballades/1/Sladek04M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/Sladek04M_annotations...,...,True,R_341_ASAP_annotation.tsv,ASAP_1_annotation.tsv,503.242188,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
275,341,R_342,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/Zhou06M.wav,{ASAP}/Chopin/Ballades/1/Zhou06M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/Zhou06M_annotations.txt,...,True,R_342_ASAP_annotation.tsv,ASAP_1_annotation.tsv,539.868490,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
374,442,R_443,Debussy,254,ASAP_Images_Book_1_1_Reflets_dans_lEau,ASAP,{ASAP}/Debussy/Images_Book_1/1_Reflets_dans_lE...,{ASAP}/Debussy/Images_Book_1/1_Reflets_dans_lE...,{ASAP}/Debussy/Images_Book_1/1_Reflets_dans_lE...,{ASAP}/Debussy/Images_Book_1/1_Reflets_dans_lE...,...,True,R_443_ASAP_annotation.tsv,ASAP_1_Reflets_dans_lEau_annotation.tsv,286.964844,validation,Images_Book_1_1_Reflets_dans_lEau,images book 1 1 reflets dans leau,debussy,debussy||images b

#### fuzzy

In [29]:
# form
FORM_REGEX = [
    ("fugue",   re.compile(r"\bfugue\b")),
    ("prelude", re.compile(r"\bprelude\b|\bpraeludium\b")),
    ("sonata", re.compile(r"\bsonata\b|\bsonatas\b")),
    ("ballades", re.compile(r"\bballades\b")),
    ("etudes", re.compile(r"\betudes\b|\bétude\b")),
    ("impromptu", re.compile(r"\bimpromptu\b")),
    ("toccata", re.compile(r"\btoccata\b")),
    ("arabeske", re.compile(r"\barabeske\b|\barabesque\b")),
    ("waltz", re.compile(r"\bwaltz\b")),
    ("nocturne", re.compile(r"\bnocturne\b")),
    ("suite", re.compile(r"\bsuite\b")),
]

def extract_form(title_norm: str) -> str:
    if not title_norm:
        return ""
    for form, rgx in FORM_REGEX:
        if rgx.search(title_norm):
            return form
    return ""

# catalogue id
NUM_RE = re.compile(r"\d+")

def extract_numbers(title_norm: str):
    if not title_norm:
        return []
    return [int(x) for x in NUM_RE.findall(title_norm)]

# composer
def composer_surname(comp_norm: str) -> str:
    if not comp_norm:
        return ""
    toks = comp_norm.split()
    return toks[-1] if toks else ""

# fuzzy
def build_block_key(title_norm: str, comp_norm: str) -> str:
    surname = composer_surname(comp_norm)
    form = extract_form(title_norm)
    nums = extract_numbers(title_norm)

    if not nums:
        return surname + "||" + form
    if not form:
        return surname + "||" + "unknown" + "||" + "||".join(map(str, nums))
    return surname + "||" + form + "||" + "||".join(map(str, nums))

In [30]:
# ASAP blocking key
df_keep["block_key"] = df_keep.apply(lambda r: build_block_key(r["title_norm"], r["comp_norm"]), axis=1)
df_asap_meta["block_key"] = df_asap_meta.apply(lambda r: build_block_key(r["title_norm"], r["comp_norm"]), axis=1)

#### case study

In [35]:
match_keys = df_asap_meta['block_key'].tolist()

In [37]:
df_asap_meta

,index,performance_id,composer,piece_id,title,source,performance_audio_external,performance_MIDI_external,MIDI_score_external,performance_annotation_external,...,aligned,performance_annotation,score_annotation,duration,split,title_drop_ASAP,title_norm,comp_norm,key,block_key
0,59,R_60,Bach,15,ASAP_Fugue_bwv_846,ASAP,{ASAP}/Bach/Fugue/bwv_846/Shi05M.wav,{ASAP}/Bach/Fugue/bwv_846/Shi05M.mid,{ASAP}/Bach/Fugue/bwv_846/midi_score.mid,{ASAP}/Bach/Fugue/bwv_846/Shi05M_annotations.txt,...,True,R_60_ASAP_annotation.tsv,ASAP_bwv_846_annotation.tsv,147.242188,train,Fugue_bwv_846,fugue bwv 846,bach,bach||fugue bwv 846,bach||fugue||846
155,214,R_215,Beethoven,78,ASAP_Piano_Sonatas_10-1,ASAP,{ASAP}/Beethoven/Piano_Sonatas/10-1/Hou02M.wav,{ASAP}/Beethoven/Piano_Sonatas/10-1/Hou02M.mid,{ASAP}/Beethoven/Piano_Sonatas/10-1/midi_score...,{ASAP}/Beethoven/Piano_Sonatas/10-1/Hou02M_ann...,...,True,R_215_ASAP_annotation.tsv,ASAP_10-1_annotation.tsv,278.940104,test,Piano_Sonatas_10-1,piano sonatas 10 1,beethoven,beethoven||piano sonatas 10 1,beethoven||sonata||10||1
269,334,R_335,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/BuiJL04M.wav,{ASAP}/Chopin/Ballades/1/BuiJL04M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/BuiJL04M_annotations.txt,...,True,R_335_ASAP_annotation.tsv,ASAP_1_annotation.tsv,558.778125,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
270,335,R_336,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/JIA06M.wav,{ASAP}/Chopin/Ballades/1/JIA06M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/JIA06M_annotations.txt,...,True,R_336_ASAP_annotation.tsv,ASAP_1_annotation.tsv,551.790625,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
271,336,R_337,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/KimSuyeon04M.wav,{ASAP}/Chopin/Ballades/1/KimSuyeon04M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/KimSuyeon04M_annotati...,...,True,R_337_ASAP_annotation.tsv,ASAP_1_annotation.tsv,528.455729,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
272,337,R_338,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/Mo07M.wav,{ASAP}/Chopin/Ballades/1/Mo07M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/Mo07M_annotations.txt,...,True,R_338_ASAP_annotation.tsv,ASAP_1_annotation.tsv,590.019531,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
273,339,R_340,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/Richardson13M.wav,{ASAP}/Chopin/Ballades/1/Richardson13M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/Richardson13M_annotat...,...,True,R_340_ASAP_annotation.tsv,ASAP_1_annotation.tsv,511.024740,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
274,340,R_341,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/Sladek04M.wav,{ASAP}/Chopin/Ballades/1/Sladek04M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/Sladek04M_annotations...,...,True,R_341_ASAP_annotation.tsv,ASAP_1_annotation.tsv,503.242188,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
275,341,R_342,Chopin,172,ASAP_Ballades_1,ASAP,{ASAP}/Chopin/Ballades/1/Zhou06M.wav,{ASAP}/Chopin/Ballades/1/Zhou06M.mid,{ASAP}/Chopin/Ballades/1/midi_score.mid,{ASAP}/Chopin/Ballades/1/Zhou06M_annotations.txt,...,True,R_342_ASAP_annotation.tsv,ASAP_1_annotation.tsv,539.868490,train,Ballades_1,ballades 1,chopin,chopin||ballades 1,chopin||ballades||1
374,442,R_443,Debussy,254,ASAP_Images_Book_1_1_Reflets_dans_lEau,ASAP,{ASAP}/Debussy/Images_Book_1/1_Reflets_dans_lE...,{ASAP}/Debussy/Images_Book_1/1_Reflets_dans_lE...,{ASAP}/Debussy/Images_Book_1/1_Reflets_dans_lE...,{ASAP}/Debussy/Images_Book_1/1_Reflets_dans_lE...,...,True,R_443_ASAP_annotation.tsv,ASAP_1_Reflets_dans_lEau_annotation.tsv,286.964844,validation,Images_Book_1_1_Reflets_dans_lEau,images book 1 1 reflets dans leau,debussy,debussy||images b

In [39]:
df_keep[df_keep['comp_norm'].str.contains('bach') & df_keep['title_norm'].str.contains('846')]

,path,metadata,mxl,pdf,version,is_user_pro,is_user_publisher,is_user_staff,has_paywall,is_rated,...,subset:rated_deduplicated,subset:no_license_conflict,subset:valid_mxl_pdf,mxl_abspath,title_norm,comp_norm,key,surname,cat_id,block_key
5528,./data/1/24/QmbHzrXuDGNG3HyzLpeBNeZ9wAj4bTDGWs...,./metadata/4/117279.json,./mxl/1/24/QmbHzrXuDGNG3HyzLpeBNeZ9wAj4bTDGWsZ...,./pdf/1/24/QmbHzrXuDGNG3HyzLpeBNeZ9wAj4bTDGWsZ...,1.14,False,False,False,False,True,...,False,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/1/24/QmbHzrXuD...,prelude i in c major bwv 846 well tempered cla...,johann sebastian bach,johann sebastian bach||prelude i in c major bw...,bach,bwv846,bach||prelude||846
31459,./data/14/35/QmWoizMg3ySMrX1uad7GQSetGEVXcpauu...,./metadata/4/130811.json,./mxl/14/35/QmWoizMg3ySMrX1uad7GQSetGEVXcpauua...,./pdf/14/35/QmWoizMg3ySMrX1uad7GQSetGEVXcpauua...,1.14,False,False,False,False,False,...,False,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/14/35/QmWoizMg...,bwv 846 the well tempered clavier part i prael...,johann sebastian bach,johann sebastian bach||bwv 846 the well temper...,bach,bwv846,bach||prelude||846
39835,./data/14/9/QmWaQcc71PMV8qF5iS9HdbLUFbWgyet6EV...,./metadata/4/1062806.json,./mxl/14/9/QmWaQcc71PMV8qF5iS9HdbLUFbWgyet6EV8...,./pdf/14/9/QmWaQcc71PMV8qF5iS9HdbLUFbWgyet6EV8...,2.06,True,False,False,False,True,...,True,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/14/9/QmWaQcc71...,bach das wohltemperierte klavier erster teil 1...,johann sebastian bach,johann sebastian bach||bach das wohltemperiert...,bach,bwv846,bach||prelude||1722||846
50094,./data/7/15/QmPdXKTQYQ5oXddUe1Rvkw13trMaDVqtpg...,./metadata/5/5650363.json,./mxl/7/15/QmPdXKTQYQ5oXddUe1Rvkw13trMaDVqtpgW...,./pdf/7/15/QmPdXKTQYQ5oXddUe1Rvkw13trMaDVqtpgW...,3.01,True,False,False,False,True,...,False,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/7/15/QmPdXKTQY...,bach johann sebastian praeludium 1 in c dur bw...,j s bach,j s bach||bach johann sebastian praeludium 1 i...,bach,bwv846,bach||prelude||1||846
52575,./data/7/52/QmPxCX56tvSF4ZTgEAGt6dFHoFo9cynYr4...,./metadata/8/5395623.json,./mxl/7/52/QmPxCX56tvSF4ZTgEAGt6dFHoFo9cynYr4B...,./pdf/7/52/QmPxCX56tvSF4ZTgEAGt6dFHoFo9cynYr4B...,2.06,False,False,False,False,True,...,True,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/7/52/QmPxCX56t...,prelude in c major bwv 846 j s bach,johann sebastian bachtranscribed by jonathan r...,johann sebastian bachtranscribed by jonathan r...,rodriguez,bwv846,rodriguez||prelude||846
108829,./data/12/24/QmUHEyeMSsaheL6iDje3iBvw1GEvDfbsn...,./metadata/0/5916116.json,./mxl/12/24/QmUHEyeMSsaheL6iDje3iBvw1GEvDfbsnA...,./pdf/12/24/QmUHEyeMSsaheL6iDje3iBvw1GEvDfbsnA...,3.01,False,False,False,False,True,...,True,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/12/24/QmUHEyeM...,preludio i bwv846 johann sebastian bach,johann sebastian bach,johann sebastian bach||preludio i bwv846 johan...,bach,bwv846,bach||unknown||846
150280,./data/0/40/Qmar5axBdWNRL7vchFDtQqo1cCaoV8XHSp...,./metadata/5/714811.json,./mxl/0/40/Qmar5axBdWNRL7vchFDtQqo1cCaoV8XHSpF...,./pdf/0/40/Qmar5axBdWNRL7vchFDtQqo1cCaoV8XHSpF...,2.06,False,False,False,False,True,...,True,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/0/40/Qmar5axBd...,bwv 846 the well tempered clavier book i i,johann sebastian bach,johann sebastian bach||bwv 846 the well temper...,bach,bwv846,bach||unknown||846
202331,./data/11/27/QmTJYYLf7PaqwbZ4h3tQ4eDnGnX9gG1SB...,./metadata/2/6060970.json,./mxl/11/27/QmTJYYLf7PaqwbZ4h3tQ4eDnGnX9gG1SBD...,./pdf/11/27/QmTJYYLf7PaqwbZ4h3tQ4eDnGnX9gG1SBD...,3.01,False,False,False,False,True,...,True,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/11/27/QmTJYYLf...,j s bach fugue in c major bwv 846 wtc 1,j s bach,j s bach||j s bach fugue in c major bwv 846 wtc 1,bach,bwv846,bach||fugue||846||1
233493,./data/6/54/QmNy4ovJ2Q4moeT2UAFr1XHnj4hcU9wfzX...,./metadata/5/130812.json,./mxl/6/54/QmNy4ovJ2Q4moeT2UAFr1XHnj4hcU9wfzXe...,./pdf/6/54/QmNy4ovJ2Q4moeT2UAFr1XHnj4hcU9wfzXe...,1.14,False,False,False,False,False,...,False,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/6/54/QmNy

In [36]:
df_keep[df_keep['block_key'].isin(match_keys)]

,path,metadata,mxl,pdf,version,is_user_pro,is_user_publisher,is_user_staff,has_paywall,is_rated,...,subset:rated_deduplicated,subset:no_license_conflict,subset:valid_mxl_pdf,mxl_abspath,title_norm,comp_norm,key,surname,cat_id,block_key
25424,./data/2/39/QmcQQkk5BGQjrjJfwYymBBvtQFdGvRZhN4...,./metadata/9/5273125.json,./mxl/2/39/QmcQQkk5BGQjrjJfwYymBBvtQFdGvRZhN4R...,./pdf/2/39/QmcQQkk5BGQjrjJfwYymBBvtQFdGvRZhN4R...,2.06,True,False,False,False,False,...,False,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/2/39/QmcQQkk5B...,promenade 1,arr maurice ravel,arr maurice ravel||promenade 1,ravel,,ravel||unknown||1
46857,./data/7/10/QmPAisUnxVQjm9G5ejKQ76pjFup5siUm2d...,./metadata/4/5353640.json,./mxl/7/10/QmPAisUnxVQjm9G5ejKQ76pjFup5siUm2da...,./pdf/7/10/QmPAisUnxVQjm9G5ejKQ76pjFup5siUm2da...,2.06,True,False,False,False,True,...,True,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/7/10/QmPAisUnx...,string quartet in f major movement 1,maurice ravel,maurice ravel||string quartet in f major movem...,ravel,,ravel||unknown||1
167413,./data/8/6/QmQ7dQTkFbV7E6hpqVgqDa6SuK6nR7NpXEH...,./metadata/3/3378576.json,./mxl/8/6/QmQ7dQTkFbV7E6hpqVgqDa6SuK6nR7NpXEHQ...,./pdf/8/6/QmQ7dQTkFbV7E6hpqVgqDa6SuK6nR7NpXEHQ...,2.06,True,False,False,False,True,...,False,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/8/6/QmQ7dQTkFb...,brahms intermezzo op 118 no 2,johannes brahms,johannes brahms||brahms intermezzo op 118 no 2,brahms,op118,brahms||unknown||118||2


### prefix 3 match

In [49]:
def prefix_n(key: str, n: int = 3) -> str:
    if not isinstance(key, str) or key == "":
        return ""
    parts = key.split("||")
    if len(parts) < n:
        return ""
    return "||".join(parts[:n])

df_keep["prefix3"] = df_keep["block_key"].apply(lambda x: prefix_n(x, 1))
df_asap_meta["prefix3"] = df_asap_meta["block_key"].apply(lambda x: prefix_n(x, 1))

In [50]:
match_keys = df_asap_meta['prefix3'].tolist()

In [51]:
df_keep['surname'][2]

'nan'

In [48]:
df_keep[(df_keep['surname'] != 'nan') & df_keep['prefix3'].isin(match_keys)]

,path,metadata,mxl,pdf,version,is_user_pro,is_user_publisher,is_user_staff,has_paywall,is_rated,...,subset:no_license_conflict,subset:valid_mxl_pdf,mxl_abspath,title_norm,comp_norm,key,surname,cat_id,block_key,prefix3
17,./data/1/11/QmbbWAiF7Eg8RapgdgwzRwgd4F4JFVmD5J...,./metadata/2/5014656.json,./mxl/1/11/QmbbWAiF7Eg8RapgdgwzRwgd4F4JFVmD5JY...,./pdf/1/11/QmbbWAiF7Eg8RapgdgwzRwgd4F4JFVmD5JY...,3.01,True,False,False,False,False,...,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/1/11/QmbbWAiF7...,urheber unbekannt,urheber unbekanntdatum in der hier transkribie...,urheber unbekanntdatum in der hier transkribie...,1796,,1796||,
18,./data/1/11/QmbbczxGc5XQjpefH9RmYzZX7qde4z3jpu...,./metadata/9/4713368.json,./mxl/1/11/QmbbczxGc5XQjpefH9RmYzZX7qde4z3jpuL...,./pdf/1/11/QmbbczxGc5XQjpefH9RmYzZX7qde4z3jpuL...,3.01,True,False,False,False,False,...,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/1/11/QmbbczxGc...,attwood together then we d fondly stray,attwood,attwood||attwood together then we d fondly stray,attwood,,attwood||,
26,./data/1/11/QmbbNbvHZ8SL65ghemEtVrQjwsRHnU5nvi...,./metadata/7/5923270.json,./mxl/1/11/QmbbNbvHZ8SL65ghemEtVrQjwsRHnU5nviR...,./pdf/1/11/QmbbNbvHZ8SL65ghemEtVrQjwsRHnU5nviR...,3.01,False,False,False,False,True,...,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/1/11/QmbbNbvHZ...,crimson peak edith s theme,fernando velazquez,fernando velazquez||crimson peak edith s theme,velazquez,,velazquez||,
29,./data/1/11/QmbbVJjMR6t54SUWCDvqrBcdGan2r2AwRm...,./metadata/2/4581256.json,./mxl/1/11/QmbbVJjMR6t54SUWCDvqrBcdGan2r2AwRm5...,./pdf/1/11/QmbbVJjMR6t54SUWCDvqrBcdGan2r2AwRm5...,3.01,True,False,False,False,False,...,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/1/11/QmbbVJjMR...,kemp s jig,anon,anon||kemp s jig,anon,,anon||,
43,./data/1/11/QmbbFTViy8S4ThhdZn3jc2Y92ZsXjK7cRM...,./metadata/2/4879688.json,./mxl/1/11/QmbbFTViy8S4ThhdZn3jc2Y92ZsXjK7cRMW...,./pdf/1/11/QmbbFTViy8S4ThhdZn3jc2Y92ZsXjK7cRMW...,3.01,True,False,False,False,False,...,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/1/11/QmbbFTViy...,col mcbain,anon,anon||col mcbain,anon,,anon||,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254055,./data/15/12/QmXBB3CGqwWUhuTMcocvhXEdeT7wc7hTZ...,./metadata/7/5920402.json,./mxl/15/12/QmXBB3CGqwWUhuTMcocvhXEdeT7wc7hTZV...,./pdf/15/12/QmXBB3CGqwWUhuTMcocvhXEdeT7wc7hTZV...,3.01,False,False,False,False,False,...,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/15/12/QmXBB3CG...,palm forced hand,palm,palm||palm forced hand,palm,,palm||,
254059,./data/15/12/QmXBrYBra51He7UCYmbp9Fccz9f7mXJPm...,./metadata/2/5136579.json,./mxl/15/12/QmXBrYBra51He7UCYmbp9Fccz9f7mXJPmm...,./pdf/15/12/QmXBrYBra51He7UCYmbp9Fccz9f7mXJPmm...,3.01,True,False,False,False,False,...,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/15/12/QmXBrYBr...,carolan lady dillon,carolan,carolan||carolan lady dillon,carolan,,carolan||,
254068,./data/15/12/QmXBXFEvupnd9E3YYjr7VewxhrfC1eGUf...,./metadata/4/5745796.json,./mxl/15/12/QmXBXFEvupnd9E3YYjr7VewxhrfC1eGUfF...,./pdf/15/12/QmXBXFEvupnd9E3YYjr7VewxhrfC1eGUfF...,3.01,True,False,False,False,False,...,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/15/12/QmXBXFEv...,triduanas a domino tommaso bai,triduanas a domino bai,triduanas a domino bai||triduanas a domino tom...,bai,,bai||,
254072,./data/15/12/QmXB3MtFhjLLfT5VzcjED73qWipct56AD...,./metadata/2/5889842.json,./mxl/15/12/QmXB3MtFhjLLfT5VzcjED73qWipct56AD5...,./pdf/15/12/QmXB3MtFhjLLfT5VzcjED73qWipct56AD5...,3.01,False,False,False,False,True,...,True,True,/mnt/ssd/hbli/datasets/PDMX/mxl/15/12/QmXB3MtF...,becrippling electrolytic unninquisitiveness,matthias djveitmanndajenth,matthias djveitmanndajenth||becrippling electr...,djveitmanndajenth,,djveitmanndajenth||,
